In [20]:
from pathlib import Path
from pydantic import BaseModel
from typing import TypedDict, Callable
import asyncio
from datetime import datetime
import aiofiles

In [21]:

class MarkDownIndexer(BaseModel):
    path: str
    headings: list[str]
    word_count: int
    last_modified: datetime

In [26]:
base = Path("/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1")
path = base / "sample_notes" / "embeddings_basics.md"

In [27]:
path

PosixPath('/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/embeddings_basics.md')

In [50]:
async def summarize_note(path: Path) -> MarkDownIndexer:
    async with aiofiles.open(path, mode= "r") as f:
        content = await f.read()

    last_modified = datetime.fromtimestamp(path.stat().st_mtime)
    word_count = len(content.split())
    lines = content.splitlines()
    headings = []
    for line in lines:
        if line.startswith("#"):
            headings.append(line.lstrip("# "))
    return MarkDownIndexer(path= str(path), headings= headings, word_count= word_count, last_modified= last_modified)





    


In [51]:
await summarize_note(path)

MarkDownIndexer(path='/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/embeddings_basics.md', headings=['Embeddings Basics', 'Why Embeddings Matter', 'Key Properties', 'Popular Models'], word_count=119, last_modified=datetime.datetime(2026, 5, 14, 14, 42, 2, 589462))

In [54]:
directory = base / "sample_notes"

In [62]:
async def main(directory: Path) -> list[MarkDownIndexer]:
    files = list(directory.glob("**/*.md"))
    return await asyncio.gather(*[summarize_note(f) for f in files])

In [64]:
result = await main(directory)

In [66]:
result

[MarkDownIndexer(path='/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/embeddings_basics.md', headings=['Embeddings Basics', 'Why Embeddings Matter', 'Key Properties', 'Popular Models'], word_count=119, last_modified=datetime.datetime(2026, 5, 14, 14, 42, 2, 589462)),
 MarkDownIndexer(path='/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/rag_overview.md', headings=['What is RAG?'], word_count=55, last_modified=datetime.datetime(2026, 5, 14, 14, 42, 12, 461746)),
 MarkDownIndexer(path='/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/vector_databases.md', headings=['Vector Databases', 'Why Not Just Use a Loop?', 'Qdrant', 'Pinecone', 'When to Use What'], word_count=117, last_modified=datetime.datetime(2026, 5, 14, 14, 42, 22, 926417))]

In [75]:
for_json = [i.model_dump() for i in result]


In [77]:
for_json

[{'path': '/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/embeddings_basics.md',
  'headings': ['Embeddings Basics',
   'Why Embeddings Matter',
   'Key Properties',
   'Popular Models'],
  'word_count': 119,
  'last_modified': datetime.datetime(2026, 5, 14, 14, 42, 2, 589462)},
 {'path': '/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/rag_overview.md',
  'headings': ['What is RAG?'],
  'word_count': 55,
  'last_modified': datetime.datetime(2026, 5, 14, 14, 42, 12, 461746)},
 {'path': '/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/vector_databases.md',
  'headings': ['Vector Databases',
   'Why Not Just Use a Loop?',
   'Qdrant',
   'Pinecone',
   'When to Use What'],
  'word_count': 117,
  'last_modified': datetime.datetime(2026, 5, 14, 14, 42, 22, 926417)}]

In [78]:
import json
json.dumps(for_json, default= str)

'[{"path": "/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/embeddings_basics.md", "headings": ["Embeddings Basics", "Why Embeddings Matter", "Key Properties", "Popular Models"], "word_count": 119, "last_modified": "2026-05-14 14:42:02.589462"}, {"path": "/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/rag_overview.md", "headings": ["What is RAG?"], "word_count": 55, "last_modified": "2026-05-14 14:42:12.461746"}, {"path": "/Volumes/MainDrive/ai engineering practice/Phase 1/Week 1/sample_notes/vector_databases.md", "headings": ["Vector Databases", "Why Not Just Use a Loop?", "Qdrant", "Pinecone", "When to Use What"], "word_count": 117, "last_modified": "2026-05-14 14:42:22.926417"}]'